# Portfolio sintetico multi-round: local greedy + SCIP

Analisi degli artifact persistiti, senza eseguire la pipeline nel notebook. L'istanza predefinita contiene 30 asset, contro i 5 degli esempi single-pass.

```bash
poetry install --with notebooks,experiments
poetry run python -m qreduce.experiments.portfolio_multi_round --run-id 004_multi_round
```

Usare `--asset-count 60` per aumentare ulteriormente la dimensione e un nuovo `--run-id` per ogni pubblicazione. SCIP richiede PySCIPOpt. Il run può terminare con una failure normalizzata se il backend non è disponibile.

**Attenzione:** tre è il numero massimo di round, non quello garantito. La pipeline si arresta già al primo successo o successo parziale; anche un problema grande può quindi produrre un solo round.

In [ ]:
from pathlib import Path

from qreduce.experiments import load_experiment_run
from qreduce.experiments.portfolio_multi_round import load_multi_round_correlation_data
from qreduce.notebooks.plots import plot_correlation_matrix
from qreduce.notebooks.presentation import display_records, display_vertical

repository_root = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / "pyproject.toml").is_file()),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from inside the QReduce repository.")

run_directory = repository_root / "outputs/004_multi_round"
artifacts = load_experiment_run(run_directory)
if not artifacts.has_reduction:
    raise ValueError("The selected artifact has no reduction rounds.")
result = artifacts.result

## 1. Configurazione, provenance e stato

In [ ]:
display_vertical(dict(artifacts.manifest), title="Manifest")
display_vertical(dict(artifacts.config["reduction"]), title="Policy di riduzione")
display_vertical(dict(result["run_result"]), title="Risultato normalizzato")
display_vertical(dict(result["metrics"]), title="Metriche complessive")

## 2. Problema e dimensioni

Le correlazioni e gli archi rappresentano una euristica di riduzione, non vincoli hard del portfolio.

In [ ]:
display_vertical({
    "asset_count": len(artifacts.problem["asset_ids"]),
    "original_variable_counts": result["original_variable_counts"],
    "residual_variable_counts": result["residual_variable_counts"],
    "round_count": len(artifacts.round_results),
    "recovery_attempt_count": len(artifacts.recovery_attempts),
}, title="Dimensioni")

## Matrice delle correlazioni degli asset

I contorni rossi indicano gli archi effettivamente persistiti per il primo round. Se la compilazione non ha prodotto un grafo, la matrice resta visibile senza contorni. Per istanze grandi i valori numerici nelle celle sono omessi per leggibilità.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

asset_ids, correlation, graph = load_multi_round_correlation_data(run_directory)
fig, ax = plot_correlation_matrix(asset_ids, correlation, graph, threshold=None)
plt.show()

## 3. Risultati per round

Ogni round presenta il proprio stato, le metriche, il dominio candidato e gli esiti MIS, solve e post-check. I campi possono essere assenti in caso di failure.

In [ ]:
for index, round_result in enumerate(artifacts.round_results):
    print(f"Round {index}")
    display_vertical(round_result["run_result"], title="Stato")
    display_vertical(round_result["metrics"], title="Metriche")
    display_vertical({
        "retained_item_ids": round_result["retained_item_ids"],
        "fixing_manifest": round_result["fixing_manifest"],
        "mis_compilation_result": round_result["mis_compilation_result"],
        "mis_execution_result": round_result["mis_execution_result"],
        "classical_result": round_result["classical_result"],
        "postcheck_result": round_result["postcheck_result"],
    }, title="Dettagli del round")

## 4. Fixing cumulativi, recovery e soluzione

Il dominio mantenuto non coincide necessariamente con la selezione finale. La riduzione euristica non certifica l’ottimo del problema originale.

In [ ]:
display_vertical({
    "final_retained_item_ids": result["final_retained_item_ids"],
    "cumulative_fixing_manifest": result["cumulative_fixing_manifest"],
    "classical_result": result["classical_result"],
    "original_problem_optimality_claim": result["original_problem_optimality_claim"],
}, title="Esito finale")
display_records(artifacts.recovery_attempts, record_label="recovery")